# Assignment 1: Fuzzy Set Operations, Cartesian Product & Max-Min Composition

In [ ]:
import numpy as np

def read_set(msg):
    vals = input(msg + " (comma separated, each between 0 and 1): ")
    arr = np.array([float(v) for v in vals.split(',')])
    if np.any(arr < 0) or np.any(arr > 1):
        raise ValueError("Membership values must be between 0 and 1")
    return arr

def read_matrix(name, rows, cols):
    print(f"Enter relation {name} ({rows} x {cols}), one row per line:")
    M = []
    for r in range(rows):
        row = [float(v) for v in input(f"  Row {r+1}: ").split(',')]
        if len(row) != cols:
            raise ValueError(f"Row must have {cols} values")
        M.append(row)
    return np.array(M)

In [ ]:
# ---------- Fuzzy set operations ----------
def union(A, B):          return np.maximum(A, B)
def intersection(A, B):   return np.minimum(A, B)
def complement(A):        return np.round(1 - A, 2)
def difference(A, B):     return np.round(np.minimum(A, 1 - B), 2)
def alg_sum(A, B):        return np.round(A + B - A * B, 2)
def alg_product(A, B):    return np.round(A * B, 2)
def bounded_sum(A, B):    return np.round(np.minimum(1, A + B), 2)
def bounded_diff(A, B):   return np.round(np.maximum(0, A - B), 2)

# ---------- Fuzzy relations ----------
def cartesian_product(P, Q):
    # R(x, y) = min(P(x), Q(y))
    R = np.zeros((len(P), len(Q)))
    for i in range(len(P)):
        for j in range(len(Q)):
            R[i, j] = min(P[i], Q[j])
    return R

def max_min_composition(R, S):
    # T(x, z) = max over y of min(R(x, y), S(y, z))
    T = np.zeros((R.shape[0], S.shape[1]))
    for i in range(R.shape[0]):
        for j in range(S.shape[1]):
            T[i, j] = np.max(np.minimum(R[i, :], S[:, j]))
    return T

In [ ]:
# Default values (can be changed from the menu)
A = np.array([0.2, 0.7, 1.0])
B = np.array([0.5, 0.4, 0.9])
R = None

while True:
    print("\n========= FUZZY SET MENU =========")
    print("A =", A, " B =", B)
    print("1. Enter fuzzy sets A and B")
    print("2. Union")
    print("3. Intersection")
    print("4. Complement of A and B")
    print("5. Difference (A - B)")
    print("6. Algebraic sum and product")
    print("7. Bounded sum and difference")
    print("8. Verify De Morgan's law")
    print("9. Fuzzy relation R by Cartesian product")
    print("10. Max-Min composition R o S")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            A = read_set("Enter set A")
            B = read_set("Enter set B")
            if len(A) != len(B):
                print("A and B must have the same number of elements. Resetting to defaults.")
                A, B = np.array([0.2, 0.7, 1.0]), np.array([0.5, 0.4, 0.9])
        elif ch == '2':
            print("A U B =", union(A, B))
        elif ch == '3':
            print("A ∩ B =", intersection(A, B))
        elif ch == '4':
            print("A' =", complement(A))
            print("B' =", complement(B))
        elif ch == '5':
            print("A - B =", difference(A, B))
        elif ch == '6':
            print("Algebraic sum     =", alg_sum(A, B))
            print("Algebraic product =", alg_product(A, B))
        elif ch == '7':
            print("Bounded sum        =", bounded_sum(A, B))
            print("Bounded difference =", bounded_diff(A, B))
        elif ch == '8':
            lhs = complement(union(A, B))
            rhs = intersection(complement(A), complement(B))
            print("(A U B)'  =", lhs)
            print("A' ∩ B'   =", rhs)
            print("De Morgan's law holds:", np.allclose(lhs, rhs))
        elif ch == '9':
            P = read_set("Enter fuzzy set P (on X)")
            Q = read_set("Enter fuzzy set Q (on Y)")
            R = cartesian_product(P, Q)
            print("Relation R = P x Q:\n", R)
        elif ch == '10':
            if R is None:
                print("First create relation R using option 9.")
                continue
            S = read_matrix("S", R.shape[1], int(input(f"Number of columns in S: ")))
            print("R =\n", R)
            print("S =\n", S)
            print("T = R o S =\n", max_min_composition(R, S))
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError as e:
        print("Input error:", e)